In [ ]:
# UNIFIED PIPELINE: MESSIDOR ONLY
# (224 Resolution, NO CLAHE, Unbalanced, Strict 80/10/10 Split)
import os
import subprocess
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive
import shutil
from google.colab import files

# 1. SETUP & PATHS
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/DR_Project"
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "DR_Messidor_Only_224_Full")
TEMP_EXTRACT_DIR = "/content/temp_raw_data_no_tf"

MESSIDOR_ZIP_001 = os.path.join(DRIVE_ROOT, "messidor_2/IMAGES.zip.001")
MESSIDOR_LABELS = os.path.join(DRIVE_ROOT, "messidor_2/messidor_data.csv")
MESSIDOR_PAIRS = os.path.join(DRIVE_ROOT, "messidor_2/messidor-2.csv")

os.makedirs(TEMP_EXTRACT_DIR, exist_ok=True)
!apt-get install -y p7zip-full

def extract_with_7z(archive_path, output_dir):
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    print(f"Extracting {archive_path}...")
    subprocess.run(['7z', 'x', archive_path, f'-o{output_dir}', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

# 2. EXTRACT RAW DATA
messidor_raw = os.path.join(TEMP_EXTRACT_DIR, "messidor")
extract_with_7z(MESSIDOR_ZIP_001, messidor_raw)
print("Extraction complete.")

# 3. PATH MAPPING & STANDARDIZATION
def get_image_map(directory):
    img_map = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                img_map[file] = os.path.join(root, file)
                img_map[os.path.splitext(file)[0]] = os.path.join(root, file)
    return img_map

messidor_img_map = get_image_map(messidor_raw)

# Parse Messidor
df_messidor_labels = pd.read_csv(MESSIDOR_LABELS, encoding='utf-8-sig').dropna(subset=['adjudicated_dr_grade'])
df_messidor_labels.columns = df_messidor_labels.columns.str.strip().str.lower()
df_messidor_pairs = pd.read_csv(MESSIDOR_PAIRS, encoding='utf-8-sig', sep=';')
df_messidor_pairs.columns = df_messidor_pairs.columns.str.strip().str.lower()
label_dict = dict(zip(df_messidor_labels['image_id'], df_messidor_labels['adjudicated_dr_grade']))

messidor_records = []
for idx, row in df_messidor_pairs.iterrows():
    pid = f"messidor_p_{idx}"
    for eye in ['left', 'right']:
        img = str(row[eye]).strip() if pd.notna(row[eye]) else None
        if img and img in messidor_img_map and img in label_dict:
            messidor_records.append({
                'patient_id': pid,
                'img_path': messidor_img_map[img],
                'label': int(label_dict[img]),
                'source': 'messidor'
            })

# 4. NO DOWNSAMPLING (Keep unbalanced)
df_final = pd.DataFrame(messidor_records).drop_duplicates(subset=['img_path']).copy()

# 5. PREPROCESSING (224 Only, NO CLAHE)
RESOLUTIONS = [224]
dir_map = {}
for res in RESOLUTIONS:
    res_base = os.path.join(OUTPUT_ROOT, f"Res_{res}")
    paths = {
        'std': os.path.join(res_base, "Messidor_Standard"),
        'split_80': os.path.join(res_base, "Split_80_10_10")
    }
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    dir_map[res] = paths

def get_cropped_padded_base(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if mask.sum() > 0:
        coords = np.argwhere(mask > 0)
        x0, y0 = coords.min(axis=0)
        x1, y1 = coords.max(axis=0) + 1
        img = img[x0:x1, y0:y1]

    h, w = img.shape[:2]
    diff = abs(h - w)
    pad_args = (diff//2, diff-(diff//2), 0, 0) if h < w else (0, 0, diff//2, diff-(diff//2))
    return cv2.copyMakeBorder(img, *pad_args, cv2.BORDER_CONSTANT, value=[0,0,0])

SEV_MAP = {0: "No", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
records = []

print(f"Extracting ROIs and generating 224 images for all {len(df_final)} samples...")
for _, row in df_final.iterrows():
    src, lbl = row['img_path'], row['label']
    s_name = f"{row['source']}_{row['patient_id']}_{os.path.basename(src)}"

    base_img = get_cropped_padded_base(src)
    if base_img is not None:
        for res in RESOLUTIONS:
            interp = cv2.INTER_AREA if res < base_img.shape[0] else cv2.INTER_CUBIC
            res_bgr = cv2.cvtColor(cv2.resize(base_img, (res, res), interpolation=interp), cv2.COLOR_RGB2BGR)

            cv2.imwrite(os.path.join(dir_map[res]['std'], s_name), res_bgr)

        records.append({
            'image_path': s_name,
            'label': lbl,
            'patient_id': row['patient_id'],
            'source_dataset': row['source'],
            'vlm_prompt': f"A fundus photograph showing {SEV_MAP[lbl]} diabetic retinopathy."
        })

df_master = pd.DataFrame(records)

# 6. STRICT SPLITTING (80/10/10 for Messidor)
print("\nCalculating splits and generating CSV...")
df_split = df_master.copy()
df_split['split'] = 'unassigned'

# Step 1: Split off 80% for Train, leaving 20% for Temp
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=42)
idx_train, idx_temp = next(gss_train.split(df_split, groups=df_split['patient_id']))

df_split.loc[df_split.iloc[idx_train].index, 'split'] = 'train'

# Step 2: Split the remaining 20% into two 50% chunks (10% Val, 10% Test)
df_temp = df_split.iloc[idx_temp].copy()
gss_val = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
idx_val, idx_test = next(gss_val.split(df_temp, groups=df_temp['patient_id']))

df_split.loc[df_temp.iloc[idx_val].index, 'split'] = 'val'
df_split.loc[df_temp.iloc[idx_test].index, 'split'] = 'test'

# Save the unified 80/10/10 CSV
csv_path = os.path.join(dir_map[224]['split_80'], "dataset_80_10_10.csv")
df_split.to_csv(csv_path, index=False)

# 7. AUTOMATIC STATS CHECK
print("\nPipeline complete. Here are your unbalanced Messidor stats:")
csv_test = pd.read_csv(csv_path)
print(f"Total Images: {len(csv_test)}")
print(f"\nLabel Distribution (Entire Dataset):\n{csv_test['label'].value_counts().sort_index().to_string()}")
print("\n--- Split Ratios ---")
print(csv_test['split'].value_counts(normalize=True).mul(100).round(2).to_string() + " %")

# 8. ZIP AND DOWNLOAD
ZIP_OUTPUT_PATH = "/content/DR_Messidor_Only_224_Full_Dataset"

print(f"\nZipping '{OUTPUT_ROOT}'...")
shutil.make_archive(ZIP_OUTPUT_PATH, 'zip', OUTPUT_ROOT)

print("Zipping complete. Triggering browser download...")
files.download(ZIP_OUTPUT_PATH + '.zip')

Mounted at /content/drive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Extracting /content/drive/MyDrive/DR_Project/messidor_2/IMAGES.zip.001...
Extraction complete.
Extracting ROIs and generating 224 images for all 1057 samples...

Calculating splits and generating CSV...

Pipeline complete. Here are your unbalanced Messidor stats:
Total Images: 1057

Label Distribution (Entire Dataset):
label
0    468
1    207
2    290
3     71
4     21

--- Split Ratios ---
split
train    80.04
test     10.03
val       9.93 %

Zipping '/content/drive/MyDrive/DR_Project/DR_Messidor_Only_224_Full'...
Zipping complete. Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# CLAHE, 512 res

In [ ]:
# UNIFIED PIPELINE: MESSIDOR ONLY
# (512 Resolution, WITH CLAHE, Unbalanced, Strict 80/10/10 Split)
import os
import subprocess
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive
import shutil
from google.colab import files

# 1. SETUP & PATHS
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/DR_Project"
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "DR_Messidor_Only_512_Full")
TEMP_EXTRACT_DIR = "/content/temp_raw_data_no_tf"

MESSIDOR_ZIP_001 = os.path.join(DRIVE_ROOT, "messidor_2/IMAGES.zip.001")
MESSIDOR_LABELS = os.path.join(DRIVE_ROOT, "messidor_2/messidor_data.csv")
MESSIDOR_PAIRS = os.path.join(DRIVE_ROOT, "messidor_2/messidor-2.csv")

os.makedirs(TEMP_EXTRACT_DIR, exist_ok=True)
!apt-get install -y p7zip-full

def extract_with_7z(archive_path, output_dir):
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    print(f"Extracting {archive_path}...")
    subprocess.run(['7z', 'x', archive_path, f'-o{output_dir}', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

# 2. EXTRACT RAW DATA
messidor_raw = os.path.join(TEMP_EXTRACT_DIR, "messidor")
extract_with_7z(MESSIDOR_ZIP_001, messidor_raw)
print("Extraction complete.")

# 3. PATH MAPPING & STANDARDIZATION
def get_image_map(directory):
    img_map = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                img_map[file] = os.path.join(root, file)
                img_map[os.path.splitext(file)[0]] = os.path.join(root, file)
    return img_map

messidor_img_map = get_image_map(messidor_raw)

# Parse Messidor
df_messidor_labels = pd.read_csv(MESSIDOR_LABELS, encoding='utf-8-sig').dropna(subset=['adjudicated_dr_grade'])
df_messidor_labels.columns = df_messidor_labels.columns.str.strip().str.lower()
df_messidor_pairs = pd.read_csv(MESSIDOR_PAIRS, encoding='utf-8-sig', sep=';')
df_messidor_pairs.columns = df_messidor_pairs.columns.str.strip().str.lower()
label_dict = dict(zip(df_messidor_labels['image_id'], df_messidor_labels['adjudicated_dr_grade']))

messidor_records = []
for idx, row in df_messidor_pairs.iterrows():
    pid = f"messidor_p_{idx}"
    for eye in ['left', 'right']:
        img = str(row[eye]).strip() if pd.notna(row[eye]) else None
        if img and img in messidor_img_map and img in label_dict:
            messidor_records.append({
                'patient_id': pid,
                'img_path': messidor_img_map[img],
                'label': int(label_dict[img]),
                'source': 'messidor'
            })

# 4. NO DOWNSAMPLING (Keep unbalanced)
df_final = pd.DataFrame(messidor_records).drop_duplicates(subset=['img_path']).copy()

# 5. PREPROCESSING (512 Only, WITH CLAHE)
RESOLUTIONS = [512]
dir_map = {}
for res in RESOLUTIONS:
    res_base = os.path.join(OUTPUT_ROOT, f"Res_{res}")
    paths = {
        'std': os.path.join(res_base, "Messidor_Standard"),
        'clahe': os.path.join(res_base, "Messidor_CLAHE"),
        'split_80': os.path.join(res_base, "Split_80_10_10")
    }
    for p in paths.values(): os.makedirs(p, exist_ok=True)
    dir_map[res] = paths

def get_cropped_padded_base(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if mask.sum() > 0:
        coords = np.argwhere(mask > 0)
        x0, y0 = coords.min(axis=0)
        x1, y1 = coords.max(axis=0) + 1
        img = img[x0:x1, y0:y1]

    h, w = img.shape[:2]
    diff = abs(h - w)
    pad_args = (diff//2, diff-(diff//2), 0, 0) if h < w else (0, 0, diff//2, diff-(diff//2))
    return cv2.copyMakeBorder(img, *pad_args, cv2.BORDER_CONSTANT, value=[0,0,0])

def apply_clahe_bgr(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

SEV_MAP = {0: "No", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
records = []

print(f"Extracting ROIs and generating 512 images for all {len(df_final)} samples...")
for _, row in df_final.iterrows():
    src, lbl = row['img_path'], row['label']
    s_name = f"{row['source']}_{row['patient_id']}_{os.path.basename(src)}"

    base_img = get_cropped_padded_base(src)
    if base_img is not None:
        for res in RESOLUTIONS:
            interp = cv2.INTER_AREA if res < base_img.shape[0] else cv2.INTER_CUBIC
            res_bgr = cv2.cvtColor(cv2.resize(base_img, (res, res), interpolation=interp), cv2.COLOR_RGB2BGR)

            # Save standard and CLAHE versions
            cv2.imwrite(os.path.join(dir_map[res]['std'], s_name), res_bgr)
            cv2.imwrite(os.path.join(dir_map[res]['clahe'], s_name), apply_clahe_bgr(res_bgr))

        records.append({
            'image_path': s_name,
            'label': lbl,
            'patient_id': row['patient_id'],
            'source_dataset': row['source'],
            'vlm_prompt': f"A fundus photograph showing {SEV_MAP[lbl]} diabetic retinopathy."
        })

df_master = pd.DataFrame(records)

# 6. STRICT SPLITTING (80/10/10 for Messidor)
print("\nCalculating splits and generating CSV...")
df_split = df_master.copy()
df_split['split'] = 'unassigned'

# Step 1: Split off 80% for Train, leaving 20% for Temp
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.80, random_state=42)
idx_train, idx_temp = next(gss_train.split(df_split, groups=df_split['patient_id']))

df_split.loc[df_split.iloc[idx_train].index, 'split'] = 'train'

# Step 2: Split the remaining 20% into two 50% chunks (10% Val, 10% Test)
df_temp = df_split.iloc[idx_temp].copy()
gss_val = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
idx_val, idx_test = next(gss_val.split(df_temp, groups=df_temp['patient_id']))

df_split.loc[df_temp.iloc[idx_val].index, 'split'] = 'val'
df_split.loc[df_temp.iloc[idx_test].index, 'split'] = 'test'

# Save the unified 80/10/10 CSV
csv_path = os.path.join(dir_map[512]['split_80'], "dataset_80_10_10.csv")
df_split.to_csv(csv_path, index=False)

# 7. AUTOMATIC STATS CHECK
print("\nPipeline complete. Here are your unbalanced Messidor stats:")
csv_test = pd.read_csv(csv_path)
print(f"Total Images: {len(csv_test)}")
print(f"\nLabel Distribution (Entire Dataset):\n{csv_test['label'].value_counts().sort_index().to_string()}")
print("\n--- Split Ratios ---")
print(csv_test['split'].value_counts(normalize=True).mul(100).round(2).to_string() + " %")

# 8. ZIP AND DOWNLOAD
ZIP_OUTPUT_PATH = "/content/DR_Messidor_Only_512_Full_Dataset"

print(f"\nZipping '{OUTPUT_ROOT}'...")
shutil.make_archive(ZIP_OUTPUT_PATH, 'zip', OUTPUT_ROOT)

print("Zipping complete. Triggering browser download...")
files.download(ZIP_OUTPUT_PATH + '.zip')

Mounted at /content/drive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Extracting /content/drive/MyDrive/DR_Project/messidor_2/IMAGES.zip.001...
Extraction complete.
Extracting ROIs and generating 512 images for all 1057 samples...

Calculating splits and generating CSV...

Pipeline complete. Here are your unbalanced Messidor stats:
Total Images: 1057

Label Distribution (Entire Dataset):
label
0    468
1    207
2    290
3     71
4     21

--- Split Ratios ---
split
train    80.04
test     10.03
val       9.93 %

Zipping '/content/drive/MyDrive/DR_Project/DR_Messidor_Only_512_Full'...
Zipping complete. Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>